
#  Construction de la variable cible (Target Engineering)

##  Contexte et objectifs

### Contexte

Dans le cadre de ce projet de maintenance prédictive, l'objectif est de développer un modèle de Machine Learning capable de détecter les périodes de fuite d'air d'un compresseur industriel.

Le jeu de données MetroPT-3 contient des mesures issues de différents capteurs, ainsi que des informations temporelles permettant de localiser les événements documentés.

### Objectifs

Ce notebook a pour objectifs :

- Définir les événements de fuite documentés.
- Construire la variable cible `target_event`.
- Associer un identifiant à chaque événement avec `event_id`.
- Vérifier la cohérence temporelle et statistique de la cible.
- Contrôler la répartition des classes positives et négatives.

###  Définition du problème

Le problème est formulé comme une classification binaire :

- Classe 0 : observation située hors des périodes de fuite documentées.
- Classe 1 : observation située dans une période de fuite documentée.

La construction de la cible repose exclusivement sur les intervalles temporels des événements définis dans le fichier de configuration du projet.


In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

# Chargement de la configuration
CHEMIN_CONFIG = Path("../config.yaml")

with open(CHEMIN_CONFIG, "r", encoding="utf-8") as fichier:
    config = yaml.safe_load(fichier)

# Chargement du dataset original
CHEMIN_DONNEES = (
    Path("../")
    / config["data"]["raw_path"]
    / config["data"]["dataset_name"]
)

df = pd.read_csv(CHEMIN_DONNEES)

print("Données chargées avec succès.")
print(f"Dimensions : {df.shape}")


Données chargées avec succès.
Dimensions : (1516948, 17)



##  Définition des événements

### Objectif

Les événements de fuite d'air sont définis à partir des intervalles temporels documentés dans le fichier `config.yaml`.

Chaque événement possède un identifiant unique, une date de début et une date de fin.

Ces intervalles serviront de référence pour construire la variable cible.


In [2]:
evenements_config = config["target"]["events"]

evenements = pd.DataFrame(evenements_config)

# Conversion des dates
evenements["start"] = pd.to_datetime(
    evenements["start"],
    errors="raise"
)

evenements["end"] = pd.to_datetime(
    evenements["end"],
    errors="raise"
)

# Vérification des intervalles
if (evenements["start"] > evenements["end"]).any():
    raise ValueError(
        "Un événement possède une date de début postérieure à sa date de fin."
    )

# Présentation
display(evenements)

print(f"Nombre d'événements documentés : {len(evenements)}")


,event_id,start,end
0,1,2020-04-18 00:00:00,2020-04-18 23:59:00
1,2,2020-05-29 23:30:00,2020-05-30 06:00:00
2,3,2020-06-05 10:00:00,2020-06-07 14:30:00
3,4,2020-07-15 14:30:00,2020-07-15 19:00:00


Nombre d'événements documentés : 4


In [3]:
evenements_tries = evenements.sort_values(
    "start"
).reset_index(drop=True)

chevauchements = []

for i in range(1, len(evenements_tries)):

    evenement_precedent = evenements_tries.iloc[i - 1]
    evenement_actuel = evenements_tries.iloc[i]

    if evenement_actuel["start"] <= evenement_precedent["end"]:

        chevauchements.append({
            "event_id_precedent": evenement_precedent["event_id"],
            "event_id_actuel": evenement_actuel["event_id"]
        })

print(f"Nombre de chevauchements : {len(chevauchements)}")

if chevauchements:
    display(pd.DataFrame(chevauchements))
else:
    print("Aucun chevauchement détecté.")


Nombre de chevauchements : 0
Aucun chevauchement détecté.



##  Construction de la variable cible

### Principe

La variable cible est construite à partir des intervalles temporels des quatre événements documentés.

Une observation est considérée comme positive lorsque son timestamp appartient à l'un des intervalles définis, bornes incluses.

- `target_event = 1` : observation appartenant à un événement documenté.
- `target_event = 0` : observation située en dehors des événements documentés.
- `event_id` : identifiant de l'événement associé à l'observation positive.

Les observations négatives ne possèdent aucun identifiant d'événement.


In [4]:
df_target = df.copy()

# Conversion du timestamp
COLONNE_TEMPORELLE = config["data"]["timestamp_column"]

df_target[COLONNE_TEMPORELLE] = pd.to_datetime(
    df_target[COLONNE_TEMPORELLE],
    errors="raise"
)

# Initialisation des variables
df_target["target_event"] = 0
df_target["event_id"] = pd.Series(
    pd.NA,
    index=df_target.index,
    dtype="Int64"
)

# Affectation des événements
for _, evenement in evenements.iterrows():

    masque_evenement = (
        (df_target[COLONNE_TEMPORELLE] >= evenement["start"])
        & (df_target[COLONNE_TEMPORELLE] <= evenement["end"])
    )

    df_target.loc[masque_evenement, "target_event"] = 1

    df_target.loc[masque_evenement, "event_id"] = int(
        evenement["event_id"]
    )

print("Construction de la cible terminée.")

Construction de la cible terminée.


In [5]:
print("Dimensions du dataset :", df_target.shape)

print("\nRépartition de la cible :")
display(
    df_target["target_event"]
    .value_counts()
    .sort_index()
    .to_frame("Nombre_observations")
)

print("\nVérification des valeurs de la cible :")
print(df_target["target_event"].value_counts(normalize=True) * 100)

print("\nNombre d'identifiants d'événements manquants :")
print(df_target["event_id"].isna().sum())

Dimensions du dataset : (1516948, 19)

Répartition de la cible :


,Nombre_observations
target_event,
0,1486994
1,29954



Vérification des valeurs de la cible :
target_event
0    98.025377
1     1.974623
Name: proportion, dtype: float64

Nombre d'identifiants d'événements manquants :
1486994


In [6]:
repartition_evenements = (
    df_target[df_target["target_event"] == 1]
    .groupby("event_id")
    .agg(
        nombre_observations=("target_event", "count"),
        date_debut=(COLONNE_TEMPORELLE, "min"),
        date_fin=(COLONNE_TEMPORELLE, "max")
    )
    .reset_index()
)

repartition_evenements["pourcentage"] = (
    repartition_evenements["nombre_observations"]
    / df_target["target_event"].sum()
) * 100

display(repartition_evenements)

print(
    "Total des observations positives :",
    repartition_evenements["nombre_observations"].sum()
)


,event_id,nombre_observations,date_debut,date_fin,pourcentage
0,1,8657,2020-04-18 00:00:01,2020-04-18 23:58:56,28.900982
1,2,2360,2020-05-29 23:30:08,2020-05-30 05:59:54,7.878747
2,3,17315,2020-06-05 10:00:04,2020-06-07 14:19:39,57.805301
3,4,1622,2020-07-15 14:30:00,2020-07-15 19:00:00,5.414970


Total des observations positives : 29954


In [7]:
# Une observation positive doit avoir un event_id
positifs_sans_id = (
    (df_target["target_event"] == 1)
    & (df_target["event_id"].isna())
).sum()

# Une observation négative ne doit pas avoir d'event_id
negatifs_avec_id = (
    (df_target["target_event"] == 0)
    & (df_target["event_id"].notna())
).sum()

# Vérification des identifiants autorisés
ids_valides = set(evenements["event_id"])

ids_observes = set(
    df_target["event_id"].dropna().astype(int).unique()
)

print("Positifs sans event_id :", positifs_sans_id)
print("Négatifs avec event_id :", negatifs_avec_id)
print("Identifiants observés :", ids_observes)

assert positifs_sans_id == 0
assert negatifs_avec_id == 0
assert ids_observes.issubset(ids_valides)

print("\nTous les contrôles de cohérence sont validés.")


Positifs sans event_id : 0
Négatifs avec event_id : 0
Identifiants observés : {np.int64(1), np.int64(2), np.int64(3), np.int64(4)}

Tous les contrôles de cohérence sont validés.


In [14]:
CHEMIN_PROCESSED = (
    Path("../")
    / config["data"]["processed_path"]
)

CHEMIN_PROCESSED.mkdir(
    parents=True,
    exist_ok=True
)

CHEMIN_SORTIE = (
    CHEMIN_PROCESSED
    / config["data"]["processed_dataset_name"]
)

df_target.to_csv(CHEMIN_SORTIE, index=False)

print(f"Dataset sauvegardé : {CHEMIN_SORTIE.resolve()}")
print(f"Dimensions : {df_target.shape}")


Dataset sauvegardé : C:\Users\ngueb\Desktop\Prédictive_maintenance_MLOPS\data\processed\MetroPT3_labeled.csv
Dimensions : (1516948, 19)


##  Conclusion

L'analyse de la variable cible `target_event` met en évidence un déséquilibre important entre les deux classes :

- **Classe 0 :** 1 486 994 observations, soit 98,03 % du jeu de données.
- **Classe 1 :** 29 954 observations, soit 1,97 % du jeu de données.

La classe positive représente les observations associées aux quatre événements de fuite d'air documentés dans le dataset MetroPT-3.

Cette distribution révèle trois points importants pour la suite du projet :

1. **Déséquilibre des classes :** un modèle qui prédit systématiquement la classe 0 pourrait obtenir une accuracy élevée sans être capable de détecter les événements de fuite.
2. **Importance de la détection :** le rappel (*Recall*) devra être particulièrement surveillé afin de limiter les événements manqués.
3. **Choix de l'évaluation :** l'accuracy ne sera pas suffisante. Nous privilégierons notamment le Recall, le F1-score, le F2-score et la Precision-Recall AUC.

Le déséquilibre devra également être pris en compte lors de la construction du protocole de validation et de l'entraînement des modèles.

**Conclusion générale :** la variable cible est correctement construite et exploitable pour la suite du projet. Toutefois, le déséquilibre des classes constitue une contrainte majeure qui devra guider les choix méthodologiques et les critères d'évaluation.

La prochaine étape consistera à approfondir l'analyse exploratoire des variables afin de mieux comprendre leurs distributions et leurs caractéristiques.